# 05 — Writeup: does the pairs-trading edge survive?

**The honest answer is that this study cannot tell, and the reason it cannot tell is the
result.**

Ten pairs, each specified from economic reasoning before it was run, each traded with the
same frozen parameters over the same decade. Out-of-sample, after costs: four make money,
six lose it. The range runs from **−42.4%** (UPS/FDX) to **+55.9%** (UNP/CSX). The
cross-sectional mean is **+0.6%**, which is indistinguishable from zero (p = 0.94), sitting
inside a standard deviation of **26 percentage points**.

So the finding is not "pairs trading works" and not "pairs trading fails". It is:

> **The dispersion across economically-similar pairs is so much larger than the average
> effect that any small-universe study of this strategy will report whichever conclusion its
> pair selection happens to produce — including this one.**

That is a claim about the epistemics of the whole genre of pairs-trading backtests, and it
is the strongest thing ten pairs can support. This notebook argues for it, and is explicit
about what the study got wrong on the way.

*Numbers are read from the same `reports/results/metrics.csv` as
`04_backtest_results.ipynb`. Tables there, argument here.*


In [ ]:
import dataclasses
import json
from pathlib import Path

import pandas as pd
from scipy import stats

from pairs_teardown.config import load_config
from pairs_teardown.data.loaders import load_or_download
from pairs_teardown.study import run_pair

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

cfg = load_config(ROOT / "configs" / "pairs.yaml")
metrics = pd.read_csv(ROOT / "reports" / "results" / "metrics.csv")
manifest = json.loads((ROOT / "reports" / "results" / "run_manifest.json").read_text())
PAIRS = [p.name for p in cfg.pairs]

net = metrics[metrics.basis == "net"].pivot_table(
    index="pair", columns="period", values="total_return") * 100
pd.set_option("display.precision", 3)


## 1. What was actually tested

Ten pairs, on log prices, over 2015-01-01 to 2024-12-31:


In [ ]:
for p in cfg.pairs:
    print(f"{p.name:<10} {p.a:<5}/{p.b:<5}  {p.rationale}")


For each pair:

1. **Spread.** A rolling 60-day OLS hedge ratio builds the spread. Rolling is used because
   static-hedge spreads fail an ADF stationarity test for WM/RSG (p ≈ 0.10) and FOXA/FOX
   (p ≈ 0.29) — the method's core assumption does not hold with a fixed ratio.
2. **Signal.** A 60-day rolling z-score of that spread. Enter at |z| ≥ 2.0, exit at
   |z| ≤ 0.5, hold the previous position in between (hysteresis, so the position does not
   churn while z hovers near a threshold).
3. **Sizing.** A **static** hedge ratio, fit by OLS **on in-sample data only** and then
   frozen — deliberately a different estimator from the signal's (see §2.1).
4. **Execution.** The position decided on day *t* is executed at day *t+1* prices.
5. **Costs.** 1 bp commission + 5 bps slippage per side, charged on `(1 + |g|)` of notional
   for every unit of position change.

Parameters were frozen before the out-of-sample period was scored, and it was scored once.
`configs/pairs.yaml` is committed, so the parameters are a record rather than a claim:


In [ ]:
for k in ["window", "entry", "exit", "signal_hedge", "sizing_hedge",
          "commission_bps", "slippage_bps", "data_start", "in_sample_end", "data_end"]:
    print(f"{k:<16} {manifest[k]}")


## 2. The five principles

### 2.1 No look-ahead bias

**What it demands.** Nothing computed for day *t* may use information from after day *t*.

**What was done.** Three mechanisms, each with a test that fails if it is removed:

| Mechanism | Guard |
|---|---|
| Positions lagged one bar before P&L | `test_engine.py` — altering a *future* price must not change any *past* P&L value |
| Sizing hedge ratio fit on in-sample data only | `test_study.py` — shocking out-of-sample prices must not move the fitted ratio, or any in-sample metric |
| Z-score uses a trailing window only | `test_spread.py` |

Both guards were checked by deliberately reintroducing the bug and confirming they go red,
because a guard that cannot fail is decoration.

**The near-miss worth recording.** An early version used the *rolling* hedge ratio for
position sizing as well as for the signal. It looked like a consistency improvement. On
synthetic data with a known true ratio it returned **−61.6%** where correct sizing returned
**+521%**: a noisy rolling estimate drifts toward zero, and a hedge ratio near zero silently
converts a market-neutral spread into an outright directional bet on one leg. This is now
blocked in three places — a runtime guard on the hedge series' standard deviation, a hard
rejection of `sizing_hedge: "rolling"` in config validation, and
`test_engine.py::test_correct_hedge_sign_neutralizes_common_move_wrong_sign_does_not`.

The general point: **a backtest that is wrong in this way looks better, not worse.** That
asymmetry is the whole argument for building the guards before trusting any number.


### 2.2 Survivorship bias — acknowledged, not engineered around

**The exposure.** All 20 tickers were chosen in 2026 from companies that still exist, still
trade, and still have a live counterpart. Pairs that would have entered a contemporaneous
2015 universe and then disappeared — through acquisition, de-listing or bankruptcy — never
had a chance to be selected. Those are disproportionately the cases where a spread diverges
and *never* reconverges, which is exactly what ruins a mean-reversion strategy.

**Why it is not corrected.** Doing so honestly needs a point-in-time constituent database
with de-listing returns (CRSP or equivalent), which this project does not have. Any cheaper
fix would be cosmetic.

**Which way it cuts.** Upward: the universe is filtered toward pairs that stayed
economically linked for a decade. The measured mean out-of-sample return of +0.6% is
therefore an *overestimate* of what a contemporaneous 2015 selection would have produced.

This mattered more when the study's six-pair version reported a clean negative result — a
bias pushing against your own conclusion is comfortable. It is less comfortable now. With a
mean near zero and four profitable pairs, **the survivorship bias is pushing in the same
direction as the more optimistic reading**, and any temptation to describe those four pairs
as evidence that the strategy works has to survive the fact that they were picked from the
set of companies that made it to 2026.


### 2.3 No data-snooping — including the part this study got wrong

**What it demands.** The strategy must not be selected using the data it is judged on —
neither the parameters nor the pairs.

**The parameters** were fixed a priori: 60 days ≈ a trading quarter, 2.0/0.5 are the
textbook bands. §2.6 shows what leaving them free would have been worth.

**The pairs are the more interesting story, because the process was flawed and the flaw is
instructive.** The universe was assembled in three waves:

| Wave | Pairs | Chosen when |
|---|---|---|
| 1 | WM/RSG, FOXA/FOX, SPY/VOO | before any backtest existed |
| 2 | KO/PEP, MA/V, XOM/CVX | a priori on economics, but run after wave 1's results were known |
| 3 | HD/LOW, UPS/FDX, UNP/CSX, DUK/SO | a priori on economics, run after waves 1–2 were known |

Every pair was specified on economic grounds before *its own* backtest was run, and no pair
has ever been removed. But it would be false to claim the whole universe was pre-registered
in one act: waves 2 and 3 were chosen by someone who already knew how wave 1 had done.

**For a period this study reported the waves as separate tiers** — wave 1 as the "official"
headline and wave 2 as a labelled "sanity check". That structure was a mistake, and the
third wave is what exposed it:


In [ ]:
WAVES = {
    "wave 1": ["WM/RSG", "FOXA/FOX", "SPY/VOO"],
    "wave 2": ["KO/PEP", "MA/V", "XOM/CVX"],
    "wave 3": ["HD/LOW", "UPS/FDX", "UNP/CSX", "DUK/SO"],
}

oos = net["out_of_sample"]
rows = []
for wave, names in WAVES.items():
    sub = oos[names]
    rows.append({
        "wave": wave,
        "pairs": len(names),
        "profitable OOS": f"{int((sub > 0).sum())} of {len(names)}",
        "mean OOS net %": sub.mean(),
        "best": f"{sub.idxmax()} ({sub.max():+.1f}%)",
        "worst": f"{sub.idxmin()} ({sub.min():+.1f}%)",
    })
display(pd.DataFrame(rows).set_index("wave").round(2))
print(f"whole universe: {int((oos > 0).sum())} of {len(oos)} profitable, "
      f"mean {oos.mean():+.2f}%")


**Wave 1 is 0 of 3. Wave 3 is 3 of 4.** Same method, same parameters, same
out-of-sample period; opposite headlines, decided entirely by which economically-plausible
pairs happened to be written down first.

Had the tiered structure survived, this project would today be reporting a confident
negative result in its headline table with a contradictory set of pairs demoted to an
appendix — and every sentence of it would have been factually true. **The two-tier structure
was a machine for producing exactly the conclusion the author reached for first.**

So the tiers are gone, and not merely from the prose. `Pair` has no `group` field,
`load_config` cannot parse one, `to_frame` emits no column that could rank pairs, and
`run_study` has no argument for running a subset —
`test_study.py::test_table_has_no_tier_column` fails if any of that comes back. The
methodological rule is enforced by the type system rather than by the analyst's memory,
which is the only version of such a rule that survives contact with a disappointing result.

**What remains uncorrected.** Waves 2 and 3 were still chosen with earlier results in
view. The defence is weak but real: selection was on economic grounds, nothing has been
dropped, and all ten are reported together. A reader who wants the strictest possible claim
should take wave 1 alone — 0 of 3, a clean negative — and note that it is a sample of three.


### 2.4 Gross and net, side by side

**What it demands.** No return is reported without its after-cost counterpart.

**What was done.** `summary()` returns `{"gross": ..., "net": ...}` as a single block, so
the two cannot be separated by accident; every row of `metrics.csv` carries a `basis`
column. Reporting gross alone is not a discipline the analyst has to remember — it is not
reachable through the API.


In [ ]:
o = metrics[metrics.period == "out_of_sample"]
g = o[o.basis == "gross"].set_index("pair").total_return * 100
n = o[o.basis == "net"].set_index("pair").total_return * 100

print(f"mean OOS total return:  gross {g.mean():+.2f}%  ->  net {n.mean():+.2f}%")
print(f"costs consume {(g.mean() - n.mean()) / g.mean():.0%} of the mean gross return\n")
print(f"profitable before costs: {int((g > 0).sum())} of {len(g)}")
print(f"profitable after costs:  {int((n > 0).sum())} of {len(n)}")
print(f"sign flipped by costs:   {sorted(g[(g > 0) & (n < 0)].index)}")


**Costs consume roughly 80% of the mean gross return** — +3.1% becomes +0.6% over three
years. That is the honest size of the friction effect, and it is large.

But the six-pair version of this study drew a stronger conclusion than the data supported.
It said costs *decide the sign*. Across ten pairs they flip only two (FOXA/FOX and MA/V),
both of which were marginal either way. UNP/CSX and DUK/SO clear their costs comfortably;
UPS/FDX and XOM/CVX were losing badly before a cent of cost was charged. **Most losing pairs
here have no gross edge, not an edge eaten by friction** — a distinction the smaller
universe hid, because with six pairs the two happened to coincide.

The one place the original claim holds exactly is **SPY/VOO**: gross Sharpe ≈ 0
out-of-sample becomes **−2.36** net. Two wrappers on the same index give a spread so tight
that the edge and its standard deviation are both tiny while the cost charge is not. The
tightest, most "reliable" pair in the study has the worst risk-adjusted outcome — the
opposite of the intuition that draws people to such pairs.


### 2.5 In-sample and out-of-sample, separated

**What it demands.** Everything estimated from data comes from the in-sample window; the
out-of-sample window is scored once, with everything frozen, and reported as it comes.

**What was done.** The split is 2021-12-31 — roughly 7 years in, 3 years out, a round date
chosen for its position in the sample rather than its effect on results. The only quantity
fit from data is the static sizing hedge ratio, fit on in-sample rows only (`study.py`) and
applied unchanged across the boundary. The split date was never swept.


In [ ]:
ins, oos_net = net["in_sample"], net["out_of_sample"]
t_stat, p_val = stats.ttest_1samp(oos_net, 0.0)
r_p, pp = stats.pearsonr(ins, oos_net)
r_s, ps = stats.spearmanr(ins, oos_net)

print("OUT-OF-SAMPLE net total return, cross-section of 10 pairs")
print(f"  mean {oos_net.mean():+.2f}%   median {oos_net.median():+.2f}%   "
      f"sd {oos_net.std():.2f}%   range {oos_net.min():+.1f}% .. {oos_net.max():+.1f}%")
print(f"  H0 mean = 0:  t = {t_stat:.3f},  p = {p_val:.3f}")
print(f"\nDoes in-sample predict out-of-sample?")
print(f"  pearson  r   = {r_p:.3f} (p = {pp:.3f})")
print(f"  spearman rho = {r_s:.3f} (p = {ps:.3f})")


**The mean out-of-sample return is statistically indistinguishable from zero** (p =
0.94), with a 26pp standard deviation around it. The median is negative while the mean is
slightly positive, because UNP/CSX carries the average by itself.

The in-sample/out-of-sample relationship sits exactly on the edge of significance:
Spearman ρ = 0.64 (p = 0.048) just clears 5%, Pearson r = 0.60 (p = 0.069) just misses. **A
result that changes verdict depending on whether you rank the data first, at n = 10, is not
a finding.** The most that can be said is that in-sample performance is probably not pure
noise as a predictor — and that this study has nowhere near the power to establish it, let
alone to justify selecting pairs on it.

Note what is *absent*: the classic overfitting signature of strong in-sample results
collapsing out-of-sample. Nothing was fit except one hedge ratio per pair, so there was
almost nothing available to overfit. The dispersion here is not overfitting. It is the
strategy's genuine pair-to-pair variance, and it is enormous.


### 2.6 What one free parameter would have been worth

The z-score window was frozen at 60. Below, the *only* thing that varies is that window —
every other parameter, the in-sample-only hedge fit, the cost model and the pair list are
untouched — and the out-of-sample net return is recomputed for all ten pairs.


In [ ]:
GRID = [40, 50, 60, 75, 90, 120]

prices_raw = load_or_download(
    list(cfg.tickers), cfg.data.start, cfg.data.end, ROOT / cfg.data.cache_dir
)

sweep = {}
for pair in cfg.pairs:
    for w in GRID:
        tuned = dataclasses.replace(cfg, signal=dataclasses.replace(cfg.signal, window=w))
        run = run_pair(pair, prices_raw, tuned)
        sweep[(pair.name, w)] = run.metrics["out_of_sample"]["net"]["total_return"] * 100

sweep_table = pd.Series(sweep).unstack().reindex(PAIRS)
sweep_table.columns.name = "z-score window"

print("NET out-of-sample total return %, varying ONLY the z-score window:")
display(sweep_table.round(1))

summary = pd.DataFrame({
    "pairs profitable": (sweep_table > 0).sum(axis=0),
    "mean across pairs %": sweep_table.mean(axis=0),
})
display(summary.T.round(2))
print("per-pair swing (max - min) across the grid, percentage points:")
display((sweep_table.max(axis=1) - sweep_table.min(axis=1)).sort_values(
    ascending=False).round(1).to_frame("swing pp"))


**Nine of the ten pairs change sign somewhere in this grid.** Only SPY/VOO, which
loses money at every window, is stable — and it is stable because its edge is too small to
be moved by anything. UPS/FDX swings 68 points and UNP/CSX 66, on a parameter no one has a
principled reason to set one way or another.

**And the frozen window is the most favourable one in the grid.** At window 60 the
cross-sectional mean is +0.64%; at all five other windows it is negative, as low as −5.15%
at window 90. This is uncomfortable and it belongs in the headline rather than a footnote:
the study's least-negative result comes from the one window that was fixed in advance. The
honest reading is that **+0.64% is the best case, not the central case** — a genuinely
pre-registered study that had happened to fix 90 instead of 60, with equal a priori
justification, would be reporting a clearly negative mean right now.

That is not an argument for switching to 90. It is the sharpest available demonstration of
the thesis: the reported answer is dominated by an arbitrary choice made before seeing any
data, which is another way of saying the study has almost no power to measure what it set
out to measure.

The correct conclusion is **not** "use whichever window looks best". It is that
**out-of-sample results this unstable under a single innocuous knob are not evidence of an
edge in either direction.** A strategy whose sign depends on the lookback is measuring
noise — which is the same conclusion §2.3 reached from pair selection and §2.5 reached from
the cross-section, arrived at three independent ways.

This sweep is reported as a finding about sensitivity. It is never used to choose a
window; 60 remains what it was before any of these numbers existed.


## 3. What would change this conclusion

1. **A genuinely large, genuinely pre-registered universe.** This is the one that matters.
   With a 26pp cross-sectional standard deviation, resolving a mean of ±2% at 95%
   confidence needs on the order of 650 pairs. Ten cannot do it; no ten can. Everything
   else below is secondary to this.
2. **Point-in-time data.** Removing the survivorship exposure of §2.2 would make the test
   fair and is expected to move the mean down.
3. **Lower costs.** Friction takes ~80% of the mean gross return, so an institution
   crossing the spread at a fraction of 6 bps per side changes the arithmetic materially —
   though it cannot fix a mean that is near zero *before* costs for most pairs.
4. **Higher frequency.** Daily bars may be too coarse for the horizon on which these
   spreads revert. Intraday data would test that, and would worsen the cost problem.
5. **Different position sizing.** Fixed unit-spread positions ignore volatility. Scaling by
   spread volatility is a real improvement and is untested here.

What would *not* change it: re-tuning the window, the bands, or the split date on this
data. §2.6 shows those choices can produce whatever headline is wanted.


## 4. Limitations

- **Ten pairs is a small cross-section**, and §2.5 is a statement about how small.
- **Waves 2 and 3 were selected with earlier results in view** (§2.3). Not fatal — nothing
  was dropped and selection was economic — but it is not the same as one-shot
  pre-registration, and it should not be described as such.
- **FOXA/FOX has 709 in-sample days** against 1,763 for every other pair, since FOX only
  lists from 2019-03-13. Its hedge ratio is fit on under half the data and all its
  statistics are noisier.
- **Costs are a flat per-side assumption.** Real slippage varies with size, volatility and
  time of day. Flat 6 bps is a reasonable retail-taker estimate on liquid US large caps,
  not a market-impact model.
- **No borrow costs, financing, or short-availability constraints.** Every pair trade is
  half short; adding these pushes results down.
- **The in-sample period contains COVID**, which dominates several pairs' spread behaviour.
- **All ten pairs are US large-cap equities** over one decade. Nothing here speaks to other
  asset classes, other regimes, or other market-cap tiers.


## 5. Conclusion

Classic pairs trading, applied to ten economically-linked US large-cap pairs with
pre-specified parameters and realistic costs, produces an out-of-sample mean return
indistinguishable from zero and a cross-sectional standard deviation of 26 percentage
points. Four pairs made money, six lost it, and the gap between best and worst is nearly
100 points. That +0.6% is itself the most favourable of six equally-defensible lookback
windows; five of the six give a negative mean.

The right conclusion is about **variance, not mean**. Three independent lines in this
notebook converge on it:

- **Pair selection** (§2.3) — the first three pairs chosen went 0 for 3; the last four went
  3 for 4.
- **Parameter choice** (§2.6) — nine of ten pairs change sign across an ordinary grid of
  lookback windows, and the pre-registered 60 turns out to be the *only* window of six with
  a positive cross-sectional mean.
- **The cross-section itself** (§2.5) — a mean of +0.6% inside a 26pp standard deviation.

Any one of those could be dismissed. Together they say the same thing: **the signal-to-noise
ratio of this strategy, at this frequency, on this kind of universe, is low enough that a
small study's conclusion is determined by its arbitrary choices rather than by the
strategy.** That is why the honest headline is not a verdict on pairs trading but a
statement about what a ten-pair backtest is capable of establishing — which is very little,
and which is worth knowing before reading the next one.

The most useful thing this project produced is not a number, it is a demonstration. At its
six-pair stage it reported a confident negative result. Four more pairs, chosen the same way
and run through identical code, moved the cross-sectional mean from clearly negative to
indistinguishable from zero. Nothing was wrong with the earlier machinery — the guards held,
the costs reconciled, the out-of-sample period was scored once. **The conclusion was simply
never as stable as the tables around it made it look**, and it took removing the
official/sanity distinction to see that, because the tiering had made the instability look
like structure.

That is the case for building research code with the discipline in the types rather than in
the prose: it does not stop you being wrong, but it makes being wrong visible from the
inside.
